In [34]:
%pip install  langchain chromadb tiktoken pypdf langchain.groq langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader

Create Documents

In [10]:
doc1= Document(
    page_content="Virat Kohli is  one of the successful cricketer in the world. He is well know for his aggressive batting style and consistency.",
    metadata={"team":"Royal Challengers Bangalore"}
)
doc2= Document(
    page_content="Rohit Sharma is  one of the successful cricketer in the world. He is well know for his aggressive batting style and he is the hitman of team India.",
    metadata={"team":"Mumbai Indians"}
)
doc3= Document(
    page_content="Jasprit Bumrah is  one of the successful cricketer in the world. He is well know for his aggressive bowling style and hardwork.",
    metadata={"team":"Mumbai Indians"}
)
doc4= Document(
    page_content="Ravindra Jadeja is  one of the successful cricketer in the world. He is well know for his aggressive all-round performance and leadership skills.",
    metadata={"team":"Chennai Super Kings"}
)
doc5= Document(
    page_content="Mahendra Singh Dhoni is  one of the successful cricketer in the world. He is well know for his aggressive batting style and consistency.  ",
    metadata={"team":"Chennai Super Kings"}
)

Merge documents in one doc list

In [11]:
docs = [doc1, doc2, doc3, doc4, doc5]

Create vector store with chroma object

In [12]:
vector_store = Chroma(
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    persist_directory='Chroma_db',
    collection_name='sample'
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_18168\3819275713.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


Add documents in vector store

In [13]:
vector_store.add_documents(docs)

['2aec90a8-0c2b-4b59-a800-5f41cc3b6588',
 'f04715c0-9926-4660-a6ac-253974dae5b0',
 '47dc9848-a03b-4066-99d7-0d20cdcdb383',
 'bac824e1-9f14-4f8a-a719-a383fa7371a3',
 '5a52d297-dc91-463a-9687-f0df2c166348']

View Documents

In [15]:
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['2aec90a8-0c2b-4b59-a800-5f41cc3b6588',
  'f04715c0-9926-4660-a6ac-253974dae5b0',
  '47dc9848-a03b-4066-99d7-0d20cdcdb383',
  'bac824e1-9f14-4f8a-a719-a383fa7371a3',
  '5a52d297-dc91-463a-9687-f0df2c166348'],
 'embeddings': array([[ 0.05504957,  0.07030239, -0.06700241, ..., -0.01293633,
          0.04944951,  0.01179109],
        [ 0.05584408,  0.01387948, -0.07358849, ..., -0.00812554,
          0.0121978 ,  0.01910035],
        [ 0.0287482 ,  0.01376123, -0.05935119, ..., -0.08319029,
          0.00716656,  0.07862164],
        [ 0.05310089,  0.04280954, -0.00961099, ..., -0.07804185,
         -0.0268048 , -0.0097186 ],
        [ 0.02742769,  0.06298593, -0.03555759, ..., -0.03514262,
          0.01315621,  0.03205595]], shape=(5, 384)),
 'documents': ['Virat Kohli is  one of the successful cricketer in the world. He is well know for his aggressive batting style and consistency.',
  'Rohit Sharma is  one of the successful cricketer in the world. He is well know for his aggr

Search Documents

In [18]:
vector_store.similarity_search(
    query="How many bowlers are there?",
    k=5
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is  one of the successful cricketer in the world. He is well know for his aggressive bowling style and hardwork.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content='Rohit Sharma is  one of the successful cricketer in the world. He is well know for his aggressive batting style and he is the hitman of team India.'),
 Document(metadata={'team': 'Chennai Super Kings'}, page_content='Mahendra Singh Dhoni is  one of the successful cricketer in the world. He is well know for his aggressive batting style and consistency.  '),
 Document(metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is  one of the successful cricketer in the world. He is well know for his aggressive batting style and consistency.'),
 Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is  one of the successful cricketer in the world. He is well know for his aggressive all-round perfor

similarity search with score

In [19]:
vector_store.similarity_search_with_score(
    query="How many bowlers are there?",
    k=5
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is  one of the successful cricketer in the world. He is well know for his aggressive bowling style and hardwork.'),
  1.2315958738327026),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content='Rohit Sharma is  one of the successful cricketer in the world. He is well know for his aggressive batting style and he is the hitman of team India.'),
  1.3621245622634888),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Mahendra Singh Dhoni is  one of the successful cricketer in the world. He is well know for his aggressive batting style and consistency.  '),
  1.415264368057251),
 (Document(metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is  one of the successful cricketer in the world. He is well know for his aggressive batting style and consistency.'),
  1.4498542547225952),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is  o

Metadata Filltering

In [21]:
vector_store.similarity_search_with_score(
    query='',
    filter={"team":"Chennai Super Kings"},
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='Mahendra Singh Dhoni is  one of the successful cricketer in the world. He is well know for his aggressive batting style and consistency.  '),
  1.8406541347503662),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is  one of the successful cricketer in the world. He is well know for his aggressive all-round performance and leadership skills.'),
  1.9512083530426025)]

Update Documents

In [28]:
Updated_Doc = Document(
    page_content="Ravindra Jadeja is  one of the successful cricketer in the world. He is well know for his aggressive all-round performance and leadership skills. He is also known as Jaddu.",
    metadata={"team":"Chennai Super Kings"}
)
vector_store.update_document(document_id='5a52d297-dc91-463a-9687-f0df2c166348', document=Updated_Doc)

In [30]:
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['2aec90a8-0c2b-4b59-a800-5f41cc3b6588',
  'f04715c0-9926-4660-a6ac-253974dae5b0',
  '47dc9848-a03b-4066-99d7-0d20cdcdb383',
  'bac824e1-9f14-4f8a-a719-a383fa7371a3',
  '5a52d297-dc91-463a-9687-f0df2c166348'],
 'embeddings': array([[ 0.05504957,  0.07030239, -0.06700241, ..., -0.01293633,
          0.04944951,  0.01179109],
        [ 0.05584408,  0.01387948, -0.07358849, ..., -0.00812554,
          0.0121978 ,  0.01910035],
        [ 0.0287482 ,  0.01376123, -0.05935119, ..., -0.08319029,
          0.00716656,  0.07862164],
        [ 0.05310089,  0.04280954, -0.00961099, ..., -0.07804185,
         -0.0268048 , -0.0097186 ],
        [ 0.01208957,  0.04637631, -0.01656109, ..., -0.07690464,
         -0.01684457, -0.01111046]], shape=(5, 384)),
 'documents': ['Virat Kohli is  one of the successful cricketer in the world. He is well know for his aggressive batting style and consistency.',
  'Rohit Sharma is  one of the successful cricketer in the world. He is well know for his aggr

Delete Document

In [32]:
vector_store.delete(ids=['47dc9848-a03b-4066-99d7-0d20cdcdb383','f04715c0-9926-4660-a6ac-253974dae5b0'])

In [33]:
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['2aec90a8-0c2b-4b59-a800-5f41cc3b6588',
  'bac824e1-9f14-4f8a-a719-a383fa7371a3',
  '5a52d297-dc91-463a-9687-f0df2c166348'],
 'embeddings': array([[ 0.05504957,  0.07030239, -0.06700241, ..., -0.01293633,
          0.04944951,  0.01179109],
        [ 0.05310089,  0.04280954, -0.00961099, ..., -0.07804185,
         -0.0268048 , -0.0097186 ],
        [ 0.01208957,  0.04637631, -0.01656109, ..., -0.07690464,
         -0.01684457, -0.01111046]], shape=(3, 384)),
 'documents': ['Virat Kohli is  one of the successful cricketer in the world. He is well know for his aggressive batting style and consistency.',
  'Ravindra Jadeja is  one of the successful cricketer in the world. He is well know for his aggressive all-round performance and leadership skills.',
  'Ravindra Jadeja is  one of the successful cricketer in the world. He is well know for his aggressive all-round performance and leadership skills. He is also known as Jaddu.'],
 'uris': None,
 'included': ['embeddings', 'document

In [ ]:
#vector_store.persist()